# EYES-DEFY-ANEMIA -- Phase 4 Classification -- new_way/ roster

8 architectures x 2 tissue types = 16 combos, added to the same shared Optuna engine
(`datapreparepipeline/trainer_engine.py`) and the same protocol as every prior combo in this
project (frozen ImageNet backbone, `Dropout->Linear` head, `dropout_rate`/`learning_rate`/
`weight_decay` all Optuna-tuned, 250-epoch ceiling, patience=7, 12-trial search) -- only the
architecture roster is new.

**5 CNN:** EfficientNet-B3 (10.70M), EfficientNet-B4 (17.55M), RegNetY-16GF (80.57M),
ConvNeXt-Base (87.57M), ConvNeXt-Large (196.23M).

**3 Hybrid:** MaxViT-Tiny (30.41M, torchvision), MaxViT-Small (68.16M, **timm**),
CoAtNet-3 (163.64M, **timm**).

**Dependency note:** MaxViT-Small and CoAtNet-3 do not exist in torchvision at all -- verified
directly (torchvision's only MaxViT variant is `maxvit_t`; CoAtNet was never ported to
torchvision in any size). `timm==1.0.28` was adopted as a real project dependency for this
reason (`classification/.project_memory/03_tech_stack_and_rules.md`), the first departure from
this project's original torchvision-only rule. **CoAtNet-3's pretrained weights are ImageNet-12k
only, with no ImageNet-1k fine-tuning stage** -- the one architecture in this roster with a
different pretraining regime than everything else in the project; worth noting explicitly if
reported alongside the other combos, not treated as an interchangeable backbone.

**Ordering:** cheapest-first by verified total parameter count, not by CNN/Hybrid family --
unlike the earlier CNN/ViT split, costs interleave here (ConvNeXt-Large at 196M is heavier than
any hybrid), so there's no clean family-based split into two sessions.

`sync_outputs()` runs after every combo, so an interrupted session still yields a downloadable
zip of everything completed so far.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 652, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 652 (delta 143), reused 289 (delta 107), pack-reused 319 (from 1)
Receiving objects: 100% (652/652), 67.41 MiB | 28.52 MiB/s, done.
Resolving deltas: 100% (306/306), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [ ]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
#
# `timm` added here -- new for this notebook. MaxViT-Small and CoAtNet-3 do
# not exist in torchvision, so this roster is the first to need it on Kaggle too.
!pip install -q optuna albumentations timm

## Data

In [5]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


## Registry sanity check

Confirms all 8 new architectures actually registered in the shared engine, and that each
builds, forward-passes, and produces the expected trainable-parameter count -- before any real
training starts. Mirrors the local structural verification already run for this roster.

In [ ]:
import sys
sys.path.insert(0, "classification/datapreparepipeline")
import torch
from trainer_engine import ARCHITECTURE_REGISTRY, DEVICE

NEW_ARCHS = ['efficientnet_b3', 'efficientnet_b4', 'regnet_y_16gf', 'convnext_base',
             'convnext_large', 'maxvit_t', 'maxvit_small', 'coatnet_3']

for arch in NEW_ARCHS:
    cfg = ARCHITECTURE_REGISTRY[arch]
    model = cfg['build_fn'](0.2).to(DEVICE)
    x = torch.randn(2, 3, cfg['input_size'], cfg['input_size']).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    assert out.shape == (2, 1), f'{arch}: bad output shape {out.shape}'
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{arch:<20} OK  out={tuple(out.shape)}  trainable_params={n_trainable}')
    del model
print('\nAll 8 new_way architectures registered and working.')

## Output syncing

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/outputs/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/new_way_results.zip. Called after EVERY training cell --
    16 combos is a long unattended run, so whatever completed so far must
    always be downloadable, same pattern as every prior notebook here."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/new_way_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

## Training -- 16 combos, cheapest architecture first

Each script's `model_name` carries a `_new_way` suffix so these results never collide with any
existing `model_name` in `classification/outputs/` (03_tech_stack_and_rules.md rule #3). Same
protocol as every other combo trained through this shared engine -- nothing about the search
itself changed, only the architecture roster.

In [ ]:
# new_way 1/16 -- efficientnet_b3, palpebral
!python classification/new_way/train_efficientnet_b3_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 2/16 -- efficientnet_b3, forniceal_palpebral
!python classification/new_way/train_efficientnet_b3_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 3/16 -- efficientnet_b4, palpebral
!python classification/new_way/train_efficientnet_b4_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 4/16 -- efficientnet_b4, forniceal_palpebral
!python classification/new_way/train_efficientnet_b4_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 5/16 -- maxvit_t, palpebral
!python classification/new_way/train_maxvit_t_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 6/16 -- maxvit_t, forniceal_palpebral
!python classification/new_way/train_maxvit_t_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 7/16 -- maxvit_small, palpebral
!python classification/new_way/train_maxvit_small_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 8/16 -- maxvit_small, forniceal_palpebral
!python classification/new_way/train_maxvit_small_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 9/16 -- regnet_y_16gf, palpebral
!python classification/new_way/train_regnet_y_16gf_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 10/16 -- regnet_y_16gf, forniceal_palpebral
!python classification/new_way/train_regnet_y_16gf_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 11/16 -- convnext_base, palpebral
!python classification/new_way/train_convnext_base_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 12/16 -- convnext_base, forniceal_palpebral
!python classification/new_way/train_convnext_base_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 13/16 -- coatnet_3, palpebral
!python classification/new_way/train_coatnet_3_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 14/16 -- coatnet_3, forniceal_palpebral
!python classification/new_way/train_coatnet_3_forniceal_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 15/16 -- convnext_large, palpebral
!python classification/new_way/train_convnext_large_palpebral_new_way.py
sync_outputs()

In [ ]:
# new_way 16/16 -- convnext_large, forniceal_palpebral
!python classification/new_way/train_convnext_large_forniceal_palpebral_new_way.py
sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever
combos completed) and zipped to `/kaggle/working/new_way_results.zip`. Both are visible in this
notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip
directly from there, or browse the folder for individual files.

Same downstream step as every prior batch in this project: extract the zip, run
`classification/v2_clean_scripts/organize_and_compare.py <path-to-extracted-outputs>` (or a
dedicated copy for this roster) to reorganize into per-combo folders and build a comparison table.

In [ ]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/new_way_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")